In [1]:
# !pip install -q -U pymupdf datasets transformers accelerate peft bitsandbytes torchao

In [2]:
import warnings
warnings.filterwarnings("ignore")


In [3]:
from dataclasses import dataclass, asdict

@dataclass
class Config:
    # Path of the pharma PDF file that will be used as the raw domain corpus.
    pdf_path: str = "/content/Nexora_Employee_Handbook_v3.1.pdf"

    # Base causal language model that we will fine-tune on pharma-domain text.
    model_name: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

    # Directory where training checkpoints will be saved during fine-tuning.
    output_dir: str = "/content/Nexora_tinyllama_lora_output"

    # Directory where the final trained LoRA adapter will be saved.
    adapter_dir: str = "/content/Nexora_tinyllama_lora_adapter"

    # Directory where cleaned and processed training data will be saved.
    processed_data_dir: str = "/Nexora/policy_processed_data"

    # Minimum paragraph length required to keep a paragraph for training.
    min_chars_per_paragraph: int = 100

    # Number of tokens in each training block for causal language modeling.
    block_size: int = 512

    # Percentage of data used for validation instead of training.
    test_size: float = 0.15

    # Random seed used to make splitting and training more reproducible.
    seed: int = 42

    # LoRA rank; controls the size and capacity of the trainable adapter.
    lora_r: int = 16

    # LoRA scaling factor; controls the strength of the LoRA update.
    lora_alpha: int = 32

    # Dropout applied inside LoRA layers to reduce overfitting.
    lora_dropout: float = 0.05

    # Number of times the model will see the complete training dataset.
    num_train_epochs: float = 3.0

    # Number of training samples processed per GPU/device at one time.
    per_device_train_batch_size: int = 1

    # Number of validation samples processed per GPU/device at one time.
    per_device_eval_batch_size: int = 1

    # Number of small batches accumulated before one optimizer update.
    gradient_accumulation_steps: int = 8

    # Step size used by the optimizer to update trainable LoRA weights.
    learning_rate: float = 2e-4

    # Fraction of early training steps used to gradually increase learning rate.
    warmup_ratio: float = 0.03

    # Regularization value used to prevent weights from becoming too large.
    weight_decay: float = 0.01

    # Number of training steps after which logs will be printed.
    logging_steps=1
    logging_first_step=True

    # Number of training steps after which validation will be performed.
    eval_steps: int = 10

    # Number of training steps after which a checkpoint will be saved.
    save_steps: int = 25

    # Maximum number of checkpoints to keep; older checkpoints will be deleted.
    save_total_limit: int = 2

    # Maximum number of training steps; -1 means train using num_train_epochs.
    max_steps: int = -1

In [4]:
config = Config()

In [5]:
import json
print(json.dumps(asdict(config), indent=2))

{
  "pdf_path": "/content/Nexora_Employee_Handbook_v3.1.pdf",
  "model_name": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
  "output_dir": "/content/Nexora_tinyllama_lora_output",
  "adapter_dir": "/content/Nexora_tinyllama_lora_adapter",
  "processed_data_dir": "/Nexora/policy_processed_data",
  "min_chars_per_paragraph": 100,
  "block_size": 512,
  "test_size": 0.15,
  "seed": 42,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "num_train_epochs": 3.0,
  "per_device_train_batch_size": 1,
  "per_device_eval_batch_size": 1,
  "gradient_accumulation_steps": 8,
  "learning_rate": 0.0002,
  "warmup_ratio": 0.03,
  "weight_decay": 0.01,
  "eval_steps": 10,
  "save_steps": 25,
  "save_total_limit": 2,
  "max_steps": -1
}


In [6]:
import os
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(config.adapter_dir, exist_ok=True)
os.makedirs(config.processed_data_dir, exist_ok=True)

In [7]:
# ============================================================
# Hugging Face repo names
# ============================================================

HF_USERNAME = "meNoodie"

BASE_MODEL_NAME = config.model_name

# Stage 1: Non-instruction LoRA adapter
HF_REPO_NON_INSTRUCTION_ADAPTER = f"{HF_USERNAME}/policy-tinyllama-non-instruction-lora-adapter"

# Stage 1 merged model
HF_REPO_NON_INSTRUCTION_MERGED = f"{HF_USERNAME}/policy-tinyllama-non-instruction-merged"

# Stage 2: Instruction LoRA adapter
HF_REPO_INSTRUCTION_ADAPTER = f"{HF_USERNAME}/policy-tinyllama-instruction-lora-adapter"

# Stage 2 merged model
HF_REPO_INSTRUCTION_MERGED = f"{HF_USERNAME}/policy-tinyllama-instruction-merged"

# Stage 3: DPO preference LoRA adapter
HF_REPO_DPO_ADAPTER = f"{HF_USERNAME}/policy-tinyllama-dpo-lora-adapter"

# Stage 3 final merged model
HF_REPO_DPO_MERGED = f"{HF_USERNAME}/policy-tinyllama-dpo-merged"

print(HF_REPO_NON_INSTRUCTION_ADAPTER)
print(HF_REPO_INSTRUCTION_ADAPTER)
print(HF_REPO_DPO_ADAPTER)

meNoodie/policy-tinyllama-non-instruction-lora-adapter
meNoodie/policy-tinyllama-instruction-lora-adapter
meNoodie/policy-tinyllama-dpo-lora-adapter


In [8]:
config.pdf_path

'/content/Nexora_Employee_Handbook_v3.1.pdf'

In [9]:
from typing import List, Dict, Any
import fitz  # PyMuPDF
def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    # Extract page-level text from a PDF.
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_index, page in enumerate(doc, start=1):
            text = page.get_text("text")
            text = text.strip() if text else ""
            if text:
                pages.append({
                    "page": page_index,
                    "text": text,
                    "char_count": len(text),
                })
    return pages

In [11]:
pdf_pages = extract_pdf_pages(config.pdf_path)

FileNotFoundError: no such file: '/content/Nexora_Employee_Handbook_v3.1.pdf'

In [ ]:
print(f"Total pages with extracted text: {len(pdf_pages)}")
print("Page-level character counts:")
for item in pdf_pages:
    print(f"Page {item['page']}: {item['char_count']} characters")

Total pages with extracted text: 31
Page-level character counts:
Page 1: 738 characters
Page 2: 467 characters
Page 3: 2367 characters
Page 4: 1677 characters
Page 5: 2855 characters
Page 6: 2460 characters
Page 7: 2810 characters
Page 8: 1649 characters
Page 9: 2810 characters
Page 10: 774 characters
Page 11: 2626 characters
Page 12: 2460 characters
Page 13: 2657 characters
Page 14: 1991 characters
Page 15: 2897 characters
Page 16: 2536 characters
Page 17: 2854 characters
Page 18: 1032 characters
Page 19: 2906 characters
Page 20: 325 characters
Page 21: 2363 characters
Page 22: 2748 characters
Page 23: 122 characters
Page 24: 2801 characters
Page 25: 2851 characters
Page 26: 3003 characters
Page 27: 3152 characters
Page 28: 3030 characters
Page 29: 2928 characters
Page 30: 3303 characters
Page 31: 1940 characters


In [ ]:
print(pdf_pages[0]["text"])

In [ ]:
import re
import unicodedata

def clean_pdf_text(text: str) -> str:
    # Standardize Unicode text so visually similar characters are treated consistently.
    # Example: "ＡＭＰＫ" becomes "AMPK" and "ﬁ" becomes "fi".
    text = unicodedata.normalize("NFKC", text)

    # Remove invisible characters that may appear during PDF text extraction.
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Join words broken by line hyphenation, e.g., "gluconeogene-\nsis" -> "gluconeogenesis".
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Replace multiple spaces/tabs with a single space.
    text = re.sub(r"[ \t]+", " ", text)

    # Convert three or more newlines into a standard paragraph break.
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove lines that contain only page numbers.
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Split text into paragraphs, clean each paragraph, and remove empty ones.
    paragraphs = []
    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        if paragraph:
            paragraphs.append(paragraph)

    # Join cleaned paragraphs with one blank line between them.
    return "\n\n".join(paragraphs)

In [ ]:
cleaned_pages = []

In [ ]:
for page in pdf_pages:
  cleaned_text = clean_pdf_text(page['text'])
  cleaned_pages.append({
      "page": page["page"],
      "text":cleaned_text,
      "char_count":len(cleaned_text)}
  )


In [ ]:
print("Total cleaned pages:", len(cleaned_pages))

Total cleaned pages: 31


In [ ]:
print(cleaned_pages[2]["text"])

In [ ]:
def split_into_paragraph_records(cleaned_pages, min_chars=80):
    paragraph_records = []

    for page in cleaned_pages:
        # Split page text into paragraphs using blank lines.
        paragraphs = page["text"].split("\n\n")

        for paragraph_index, paragraph in enumerate(paragraphs, start=1):
            # Remove extra spaces from the beginning and end.
            paragraph = paragraph.strip()

            # Skip very short paragraphs because they are usually headings, page numbers, or noise.
            if len(paragraph) < min_chars:
                continue

            # Store each useful paragraph with basic metadata.
            paragraph_records.append({
                "text": paragraph,
                "source_page": page["page"],
                "paragraph_id": paragraph_index,
                "char_count": len(paragraph),
            })

    return paragraph_records

In [ ]:
paragraph_records = split_into_paragraph_records(cleaned_pages)

In [ ]:
print("Total paragraph records:", len(paragraph_records))

In [ ]:
for record in paragraph_records[:3]:
    print("=" * 80)
    print(f"Page: {record['source_page']} | Paragraph: {record['paragraph_id']} | Characters: {record['char_count']}")
    print(record["text"])

Page: 1 | Paragraph: 3 | Characters: 603
CONFIDENTIALITY NOTICE This document is the exclusive property of Nexora Technologies Pvt. Ltd. and contains proprietary, confidential, and legally privileged information. It is intended solely for the use of current employees of Nexora Technologies Pvt. Ltd. Unauthorized reproduction, distribution, disclosure, or use of any portion of this handbook — in whole or in part — is strictly prohibited and may constitute a violation of applicable law and company policy. If you have received this document in error, please notify the Human Resources department immediately and return or destroy all copies.
Page: 2 | Paragraph: 1 | Characters: 455
Table of Contents Section 1 — Welcome to Nexora Technologies Section 2 — Employment Policies Section 3 — Code of Conduct Section 4 — Attendance and Working Hours Section 5 — Leave Policies Section 6 — Compensation and Benefits Section 7 — Information Security Policies Section 8 — Compliance Policies Section 9 — P

In [ ]:

raw_pages_path = os.path.join(config.processed_data_dir, "pdf_pages_raw.jsonl")
paragraphs_path = os.path.join(config.processed_data_dir, "pharma_paragraph_process.jsonl")

with open(raw_pages_path, "w", encoding="utf-8") as f:
    for item in pdf_pages:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(paragraphs_path, "w", encoding="utf-8") as f:
    for item in paragraph_records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved raw pages to: {raw_pages_path}")
print(f"Saved cleaned paragraph corpus to: {paragraphs_path}")

Saved raw pages to: /Nexora/policy_processed_data/pdf_pages_raw.jsonl
Saved cleaned paragraph corpus to: /Nexora/policy_processed_data/pharma_paragraph_process.jsonl


In [ ]:
from datasets import Dataset
if len(paragraph_records) < 2:
    raise ValueError(
        "The extracted corpus is too small. Please provide a larger pharma PDF or lower min_chars_per_paragraph."
    )
text_dataset = Dataset.from_list(paragraph_records)

In [ ]:
print(text_dataset)

In [ ]:
print(text_dataset[0])

In [ ]:
split_dataset = text_dataset.train_test_split(test_size=config.test_size, seed=config.seed)

from datasets import DatasetDict
dataset = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 27
    })
    validation: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 5
    })
})


# ====================================
#  Load tokenizer
# ====================================


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

# Some Llama-style models do not define a pad token.
# For causal LM fine-tuning, using EOS as PAD is a common practical choice.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

In [ ]:
# ============================================================
# 12. Tokenization and text packing
# ============================================================

def tokenize_function(examples):
    # Tokenize text without padding. Padding is handled dynamically by the collator.
    return tokenizer(examples["text"])

In [ ]:
tokenized_datasets = dataset.map(
    tokenize_function,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing text corpus",
)

Tokenizing text corpus:   0%|          | 0/27 [00:00<?, ? examples/s]

Tokenizing text corpus:   0%|          | 0/5 [00:00<?, ? examples/s]

In [ ]:
def create_training_blocks(tokenized_examples):
    # Join all token IDs from multiple examples into one long list.
    all_input_ids = []
    all_attention_masks = []

    for input_ids in tokenized_examples["input_ids"]:
        all_input_ids.extend(input_ids)

    for attention_mask in tokenized_examples["attention_mask"]:
        all_attention_masks.extend(attention_mask)

    # Calculate how many complete blocks we can create.
    total_tokens = len(all_input_ids)
    usable_tokens = (total_tokens // config.block_size) * config.block_size

    # If we do not have enough tokens to create even one block, return empty data.
    if usable_tokens == 0:
        return {
            "input_ids": [],
            "attention_mask": [],
            "labels": [],
        }

    # Keep only tokens that can fit into complete fixed-size blocks.
    all_input_ids = all_input_ids[:usable_tokens]
    all_attention_masks = all_attention_masks[:usable_tokens]

    # Split the long token list into fixed-size training blocks.
    input_id_blocks = []
    attention_mask_blocks = []

    for start_index in range(0, usable_tokens, config.block_size):
        end_index = start_index + config.block_size

        input_id_blocks.append(all_input_ids[start_index:end_index])
        attention_mask_blocks.append(all_attention_masks[start_index:end_index])

    # For causal language modeling, labels are the same as input IDs.
    # The model uses these labels to learn next-token prediction.
    labels = input_id_blocks.copy()

    return {
        "input_ids": input_id_blocks,
        "attention_mask": attention_mask_blocks,
        "labels": labels,
    }

In [ ]:
final_dataset = tokenized_datasets.map(
    create_training_blocks,
    batched=True,
    desc=f"Creating fixed-size training blocks of {config.block_size} tokens",
)

Creating fixed-size training blocks of 512 tokens:   0%|          | 0/27 [00:00<?, ? examples/s]

Creating fixed-size training blocks of 512 tokens:   0%|          | 0/5 [00:00<?, ? examples/s]

In [ ]:
import torch
use_cuda = torch.cuda.is_available()
print("CUDA available:", use_cuda)
if use_cuda:
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: False


In [ ]:
# Clear memory before loading the model.
import gc
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

In [ ]:
from transformers import AutoModelForCausalLM

if use_cuda:
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training

    # Configure 4-bit quantization to reduce GPU memory usage.
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # Load the base model in 4-bit mode on available GPU devices.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Prepare the quantized model for stable LoRA/QLoRA training.
    base_model = prepare_model_for_kbit_training(base_model)

else:
    # Load the base model normally when GPU is not available.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

# Disable cache during training to reduce memory usage and avoid training warnings.
base_model.config.use_cache = False

print("Base model loaded successfully.")

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Base model loaded successfully.


In [ ]:
from peft import LoraConfig
from peft import TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


In [ ]:
from peft import get_peft_model
model = get_peft_model(base_model, lora_config)

In [ ]:
model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [ ]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
from transformers import TrainingArguments

In [ ]:
training_kwargs = dict(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_steps=5,
    weight_decay=config.weight_decay,

    # Log training loss at every step for small demo datasets.
    logging_steps=1,
    logging_first_step=True,

    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    fp16=use_cuda,
    bf16=False,
    report_to="none",
    remove_unused_columns=False,
)

In [ ]:
from transformers import TrainingArguments
training_args = TrainingArguments(**training_kwargs)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_dataset["train"],
    eval_dataset=final_dataset["validation"],
    data_collator=data_collator,
)
print("Trainer is ready.")

Trainer is ready.


In [ ]:
# 18. Start training
# ============================================================
# train_result = trainer.train()
print("Training completed.")

Step,Training Loss
1,2.016984
2,2.089485
3,2.045815
4,1.987251
5,1.902054
6,1.947178
7,1.883570
8,2.227789
9,1.773493
10,1.833216


Training completed.


In [ ]:
for log in trainer.state.log_history:
    print(log)

{'loss': 2.016983985900879, 'grad_norm': 0.5033771991729736, 'learning_rate': 0.0, 'epoch': 0.3076923076923077, 'step': 1}
{'loss': 2.0894851684570312, 'grad_norm': 0.5244603157043457, 'learning_rate': 4e-05, 'epoch': 0.6153846153846154, 'step': 2}
{'loss': 2.0458147525787354, 'grad_norm': 0.6097139120101929, 'learning_rate': 8e-05, 'epoch': 0.9230769230769231, 'step': 3}
{'loss': 1.987250804901123, 'grad_norm': 0.7575061917304993, 'learning_rate': 0.00012, 'epoch': 1.0, 'step': 4}
{'loss': 1.9020538330078125, 'grad_norm': 0.5186704397201538, 'learning_rate': 0.00016, 'epoch': 1.3076923076923077, 'step': 5}
{'loss': 1.9471783638000488, 'grad_norm': 0.5644506812095642, 'learning_rate': 0.0002, 'epoch': 1.6153846153846154, 'step': 6}
{'loss': 1.8835700750350952, 'grad_norm': 0.5473470687866211, 'learning_rate': 0.00017142857142857143, 'epoch': 1.9230769230769231, 'step': 7}
{'loss': 2.2277891635894775, 'grad_norm': 1.000307321548462, 'learning_rate': 0.00014285714285714287, 'epoch': 2.0,

In [ ]:
trainer.model.save_pretrained(config.adapter_dir)
tokenizer.save_pretrained(config.adapter_dir)

('/content/Nexora_tinyllama_lora_adapter/tokenizer_config.json',
 '/content/Nexora_tinyllama_lora_adapter/tokenizer.json')

In [ ]:
print(f"LoRA adapter saved to: {config.adapter_dir}")
print("Saved files:")
print(os.listdir(config.adapter_dir))

LoRA adapter saved to: /content/Nexora_tinyllama_lora_adapter
Saved files:
['tokenizer_config.json', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer.json', 'README.md']


In [ ]:

del trainer

try:
    del model
    del base_model
except NameError:
    pass

gc.collect()

if use_cuda:
    torch.cuda.empty_cache()

In [ ]:
from transformers import AutoTokenizer
inference_tokenizer = AutoTokenizer.from_pretrained(config.adapter_dir, use_fast=True)

if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

In [ ]:
if use_cuda:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
from peft import PeftModel
inference_model = PeftModel.from_pretrained(inference_base_model, config.adapter_dir)

In [ ]:
inference_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [ ]:
print("Base model + LoRA adapter loaded successfully for inference.")

Base model + LoRA adapter loaded successfully for inference.


In [ ]:
# ============================================================
# 22. Inference helper
# ============================================================
# Since this is non-instruction fine-tuning, prompts should look like text continuations,
# not chat-style questions.

def generate_completion(prompt: str, max_new_tokens: int = 120) -> str:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Convert prompt text into token IDs.
    inputs = inference_tokenizer(prompt, return_tensors="pt").to(device)

    # Generate text without calculating gradients because we are doing inference, not training.
    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=inference_tokenizer.eos_token_id,
        )

    # Convert generated token IDs back into readable text.
    return inference_tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
prompts = [
"Our company culture is defined by rigorous engineering and" ,
"The core values of Nexora Technologies include Integrity First, Customer Obsession, Intellectual Courage, Ownership Mentality, and ",
"Nexora Technologies Pvt. Ltd. is a company that focuses on"
]


In [ ]:
prompts

In [ ]:
prompts = ["Section 1.3 of the Nexora Technologies handbook defines Ownership Mentality as: Every Nexora employee is expected to act as a steward of the company's mission, resources, and reputation. Therefore, an employee with Ownership Mentality will"]

In [ ]:
for prompt in prompts:
    print("=" * 100)
    print("PROMPT:")
    print(prompt)
    print("\nMODEL CONTINUATION:")
    print(generate_completion(prompt, max_new_tokens=120))
    print()

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT:
Our company culture is defined by rigorous engineering and

MODEL CONTINUATION:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Our company culture is defined by rigorous engineering and customer focus. We are committed to continual improvement through the application of the latest technologies, lean manufacturing practices, and agile software development methodology. We welcome all ideas that can drive our business forward.
Experience in the following languages: Python, Java, C#, Bash, Perl, PHP, Ruby, Kotlin, Go, Scala, Javascript, Rust, Golang, Node JS, TypeScript, C++, Haskell, XML/XSD, JSON, SQL, NoSQL, MongoDB, Elasticsearch, PostgreSQL, MySQL, Oracle, SAP

PROMPT:
The core values of Nexora Technologies include Integrity First, Customer Obsession, Intellectual Courage, Ownership Mentality, and 

MODEL CONTINUATION:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The core values of Nexora Technologies include Integrity First, Customer Obsession, Intellectual Courage, Ownership Mentality, and 10x Business.
This position is responsible for creating detailed technical documentation of the products that are deployed on a daily basis in production and ensure that all documentation is maintained according to the standard set by the Product Documentation team.
The ideal candidate should have experience with Agile methodology, Scrum, Kanban, and SCRUMBOOK. Experience with AWS, Terraform, Ansible, Git, and GitHub is highly desirable.
Nexora Technologies is a global provider of enterprise-grade network security solutions that protect over 200 customers from advanced

PROMPT:
Nexora Technologies Pvt. Ltd. is a company that focuses on

MODEL CONTINUATION:
Nexora Technologies Pvt. Ltd. is a company that focuses on the delivery of IT services to large enterprises and government organizations in India, North America, Europe and Australia. The company's techno

## Push the model to Hugging Face Hub

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# Push the LoRA adapter to Hugging Face
inference_model.push_to_hub(
    HF_REPO_NON_INSTRUCTION_ADAPTER,
    private=True,
    token=os.environ.get("HF_TOKEN"), # Optional: if not set, it will use the token from notebook_login()
)

# Push the tokenizer to Hugging Face
inference_tokenizer.push_to_hub(
    HF_REPO_NON_INSTRUCTION_ADAPTER,
    private=True,
    token=os.environ.get("HF_TOKEN"), # Optional: if not set, it will use the token from notebook_login()
)

print(f"LoRA adapter and tokenizer pushed to: https://huggingface.co/{HF_REPO_NON_INSTRUCTION_ADAPTER}")

# **SFT BASED TUNING**

In [ ]:
instruction_path = "/content/nexora_sft_combined.jsonl"

In [ ]:
from datasets import load_dataset

In [ ]:
instruction_dataset = load_dataset(
    "json",
    data_files=instruction_path,
    split="train"
)

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
def format_instruction_record(record):
    instruction = str(record.get("instruction", "")).strip()
    input_text = str(record.get("input", "")).strip()
    output_text = str(record.get("output", "")).strip()

    if input_text:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n{output_text}"
        )
    else:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{output_text}"
        )

    return {"text": text}

In [ ]:
instruction_dataset = instruction_dataset.map(format_instruction_record)

Map:   0%|          | 0/164 [00:00<?, ? examples/s]

In [ ]:
instruction_datasets = instruction_dataset.train_test_split(
    test_size=0.15,
    seed=42
)

instruction_datasets["validation"] = instruction_datasets.pop("test")

print(instruction_datasets)
print("Train examples:", len(instruction_datasets["train"]))
print("Validation examples:", len(instruction_datasets["validation"]))

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 139
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 25
    })
})
Train examples: 139
Validation examples: 25


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(tokenizer.pad_token)

</s>


In [ ]:
def tokenize_instruction_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )

    # For causal LM, labels are copied from input_ids.
    tokens["labels"] = tokens["input_ids"].copy()

    # Ignore padding tokens in the loss calculation.
    tokens["labels"] = [
        [
            token if mask == 1 else -100
            for token, mask in zip(input_ids, attention_mask)
        ]
        for input_ids, attention_mask in zip(tokens["input_ids"], tokens["attention_mask"])
    ]

    return tokens

In [ ]:
instruction_tokenized_datasets = instruction_datasets.map(
    tokenize_instruction_function,
    batched=True,
    remove_columns=instruction_datasets["train"].column_names,
    desc="Tokenizing instruction dataset",
)

print(instruction_tokenized_datasets)

Tokenizing instruction dataset:   0%|          | 0/139 [00:00<?, ? examples/s]

Tokenizing instruction dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 139
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 25
    })
})


In [ ]:
# ============================================================
# Load merged Stage 1 model and add new LoRA adapter for instruction tuning
# ============================================================

# Merged Stage 1 Model
#    +
# New LoRA adapter for instruction tuning

import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

use_cuda = torch.cuda.is_available()

merged_model_dir = "/content/Nexora_tinyllama_lora_adapter"
if use_cuda:
    # Load merged Stage 1 model in 4-bit mode for QLoRA instruction tuning.
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )

    instruction_base_model = prepare_model_for_kbit_training(instruction_base_model)

else:
    # CPU fallback. Training on CPU will be slow.
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

instruction_base_model.config.use_cache = False



OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/Nexora_tinyllama_lora_adapter'. Use `repo_type` argument if needed.

In [ ]:
# Create a new LoRA adapter for instruction fine-tuning.
instruction_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

instruction_model = get_peft_model(
    instruction_base_model,
    instruction_lora_config
)

instruction_model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [ ]:
from transformers import DataCollatorForLanguageModeling
instruction_data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

In [ ]:
instruction_output_dir =r"D:\code.folder\Fullstack-AI\FineTune_Project\Data\Nexora_tinyllama_instruction_lora_output"
instruction_adapter_dir = r"D:\code.folder\Fullstack-AI\FineTune_Project\Data\Nexora_tinyllama_instruction_lora_output_tinyllama_instruction_lora_adapter"

# os.makedirs(instruction_output_dir, exist_ok=True)
# os.makedirs(instruction_adapter_dir, exist_ok=True)

In [ ]:
from transformers import TrainingArguments

instruction_training_args = TrainingArguments(
    output_dir=instruction_output_dir,

    # Train for 5 full epochs.
    num_train_epochs=5,
    max_steps=-1,

    # Batch settings.
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    # Optimizer settings.
    learning_rate=1e-4,
    warmup_steps=5,
    weight_decay=0.01,

    # Show training loss at every step.
    logging_steps=1,
    logging_first_step=True,

    # Run validation at every step.
    eval_strategy="steps",
    eval_steps=1,

    # Save checkpoints.
    save_steps=25,
    save_total_limit=2,

    # Precision settings.
    fp16=use_cuda,
    bf16=False,

    # Disable external logging tools.
    report_to="none",

    # Keep required columns.
    remove_unused_columns=False,
)

print(instruction_training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=1,
eval_strategy=IntervalStrategy.STEPS,
eval_us

In [ ]:
from transformers import Trainer
instruction_trainer = Trainer(
    model=instruction_model,
    args=instruction_training_args,
    train_dataset=instruction_tokenized_datasets["train"],
    eval_dataset=instruction_tokenized_datasets["validation"],
    data_collator=instruction_data_collator,
)

print("Instruction Trainer is ready.")

Instruction Trainer is ready.


In [ ]:
instruction_train_result = instruction_trainer.train()

Step,Training Loss,Validation Loss
1,2.467456,2.484206
2,2.449810,2.470338
3,2.457156,2.440003
4,2.442057,2.395418
5,2.282512,2.336927
6,2.236941,2.266110
7,2.112822,2.197303
8,2.192437,2.134722
9,2.124740,2.080491
10,1.948635,2.029899


In [ ]:
print("Instruction fine-tuning completed.")
print(instruction_train_result)

Instruction fine-tuning completed.
TrainOutput(global_step=90, training_loss=1.5832306583722433, metrics={'train_runtime': 1105.7625, 'train_samples_per_second': 0.629, 'train_steps_per_second': 0.081, 'total_flos': 2235660301762560.0, 'train_loss': 1.5832306583722433, 'epoch': 5.0})


In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, BitsAndBytesConfig



if torch.cuda.is_available():
    torch.cuda.empty_cache()

if use_cuda:
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

final_instruction_model = PeftModel.from_pretrained(
    base_model,
    instruction_adapter_dir,
)

final_instruction_model.eval()

print("Final instruction-tuned model loaded successfully.")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5037.50it/s]


Final instruction-tuned model loaded successfully.


In [ ]:
def build_instruction_prompt(instruction, input_text=""):
    instruction = instruction.strip()
    input_text = input_text.strip()

    if input_text:
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n"
        )

    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n"
    )


def generate_instruction_response(instruction, input_text="", max_new_tokens=150):
    prompt = build_instruction_prompt(instruction, input_text)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(final_instruction_model.device)

    with torch.no_grad():
        outputs = final_instruction_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
test_questions = [
    "What are the core values of Nexora Technologies?"]
for question in test_questions:
    print("=" * 100)
    print("QUESTION:")
    print(question)

    print("\nMODEL RESPONSE:")
    print(generate_instruction_response(question, max_new_tokens=150))

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
What are the core values of Nexora Technologies?

MODEL RESPONSE:
### Instruction:
What are the core values of Nexora Technologies?

### Response:
I have to write a paragraph with the following guidelines:
- 100 words or less.
- Paragraph should be indented using 1 space per indent and no more than one line break.
- Use bold, italics, and capitalized text.
- Use all caps for emphasis.
- No bold text in your responses.
- Be concise and professional.

#### Examples ##################################################################

### Question 1 ##################################################################

> I agree that Nexora Technologies follows the core values outlined in the “Code of Conduct” and is committed to fostering an inclusive environment where every individual


: 